# Baseline — Depth 6

Launches the `baseline` experiment for every seed-triple index in
`configs/baseline_6layer.yaml` and summarises the completed runs. Results are written to
`results/baseline/depth_6/seed_<N>/metrics.json`.


In [1]:
import os
import sys
from pathlib import Path


# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


'1'

## Run

Executes the runner once per seed-triple index, skipping
any seed whose `metrics.json` already exists.

In [ ]:
import re
import subprocess
import time
from pathlib import Path

import yaml
from tqdm.auto import tqdm


CONFIG = "configs/baseline_6layer.yaml"
APPROACH = "baseline"
DEVICE = "gpu"  # Use "cpu" or "auto" when needed.
results_dir = Path("results/baseline/depth_6")

run_env = os.environ.copy()
run_env["TF_CPP_MIN_LOG_LEVEL"] = "2"
if DEVICE == "gpu":
    run_env["CUDA_VISIBLE_DEVICES"] = "0"
    run_env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
elif DEVICE == "cpu":
    run_env["CUDA_VISIBLE_DEVICES"] = ""

config_data = yaml.safe_load(Path(CONFIG).read_text())
n_indices = int(config_data["seeds"]["seed_triples"])
total_updates = int(config_data["training"]["total_updates"])
checkpoint_frequency = int(config_data["output"]["checkpoint_frequency"])
print(f"Baseline depth 6 | device={DEVICE} | seeds={n_indices} | updates/seed={total_updates}")

# track the duration of each seed run and estimate remaining time with tqdm
def format_duration(seconds):
    seconds = max(0, int(seconds))
    return f"{seconds // 60}m {seconds % 60:02d}s"


def checkpoint_step(checkpoint_dir):
    """Infer completed updates from the latest saved checkpoint number."""
    state_file = checkpoint_dir / "checkpoint"
    if not state_file.exists() or checkpoint_frequency <= 0:
        return 0
    matches = re.findall(r'ckpt-(\d+)', state_file.read_text())
    if not matches:
        return 0
    return min(int(matches[-1]) * checkpoint_frequency, total_updates)

seed_durations = []
experiment_started = time.monotonic()
completed_seeds = 0

for i in range(n_indices):
    seed_dir = results_dir / f"seed_{i}"
    metrics_path = seed_dir / "metrics.json"
    checkpoint_dir = seed_dir / "checkpoint"
    log_path = seed_dir / "run.log"

    if metrics_path.exists():
        completed_seeds += 1
        tqdm.write(f"[skip] seed {i:02d} already complete")
        continue

    seed_dir.mkdir(parents=True, exist_ok=True)
    tqdm.write(f"[run ] seed {i:02d}/{n_indices - 1:02d} started")
    command = [
        sys.executable, "-c",
        "import tensorflow as tf; "
        "tf.get_logger().setLevel('ERROR'); "
        "from experiments.run_baseline import main; main()",
        CONFIG, "--seed-index", str(i),
    ]
    started = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            env=run_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )
        progress = tqdm(
            total=total_updates,
            initial=checkpoint_step(checkpoint_dir),
            desc=f"seed {i:02d}",
            unit="step",
            dynamic_ncols=True,
            bar_format=(
                "{desc} |{bar:28}| {percentage:3.0f}% "
                "[{n_fmt}/{total_fmt}] [{elapsed}<{remaining}, {rate_fmt}]"
            ),
        )
        last_step = progress.n

        while process.poll() is None:
            current_step = checkpoint_step(checkpoint_dir)
            if current_step > last_step:
                progress.update(current_step - last_step)
                progress.set_postfix_str(f"checkpoint {current_step}/{total_updates}")
                last_step = current_step
            time.sleep(1)

        return_code = process.wait()
        final_step = checkpoint_step(checkpoint_dir)
        if return_code == 0 and final_step < total_updates:
            progress.update(total_updates - final_step)
        progress.close()

    elapsed = time.monotonic() - started
    if return_code != 0:
        log_tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-12:]
        tqdm.write(f"[fail] seed {i:02d} failed; log: {log_path}")
        print("\n".join(log_tail))
        raise subprocess.CalledProcessError(return_code, command)

    seed_durations.append(elapsed)
    completed_seeds += 1
    average_seed = sum(seed_durations) / len(seed_durations)
    remaining_seeds = n_indices - completed_seeds
    tqdm.write(
        f"[done] seed {i:02d} | time={format_duration(elapsed)} | "
        f"remaining={format_duration(average_seed * remaining_seeds)} | log={log_path}"
    )

print(
    f"Complete: {completed_seeds}/{n_indices} seeds | "
    f"total time={format_duration(time.monotonic() - experiment_started)}",
    flush=True, 
)



Baseline depth 6 | device=gpu | seeds=20 | updates/seed=2500
[skip] seed 00 already complete
[run ] seed 01/19 started


seed 01 |###########2                |  40% [1000/2500] [00:00<?, ?step/s]

## Summary

In [ ]:
import json

results_dir = Path("results/baseline/depth_6")
rows = []
for seed_dir in sorted(results_dir.glob("seed_*")):
    metrics = json.loads((seed_dir / "metrics.json").read_text())
    diag = metrics.get("training_diagnostic", {})
    rows.append((
        metrics.get("seed_index"),
        metrics["test_acc"],
        metrics["test_loss"],
        metrics.get("n_parameters"),
        diag.get("mean_param_grad_variance"),
    ))
 
rows.sort()
if rows:
    print(f"{len(rows)} completed runs in {results_dir}:")
    print(f"{'seed':>4}  {'test_acc':>8}  {'test_loss':>9}  "
          f"{'n_params':>8}  {'grad_var':>12}")
    for seed, acc, loss, nparams, gv in rows:
        print(f"{seed:>4}  {acc:>8.4f}  {loss:>9.4f}  "
              f"{nparams:>8}  {gv:>12.3e}")
else:
    print(f"No completed  runs found in {results_dir}.")
